In [18]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import h5py
from collections import Counter
import scipy.sparse as sp
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patheffects as path_effects
from matplotlib_venn import venn3
import pyranges as pr
import yaml
import time
from itertools import combinations
import gget

from pycirclize import Circos

import anndata as an
import scanpy as sc
import rapids_singlecell as rsc

import scvi
from scvi.external import SysVI

import cupy as cp
import cudf
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

from cuml.manifold.umap import simplicial_set_embedding
from scanpy.tools._utils import get_init_pos_from_paga 
from cuml.manifold.umap import find_ab_params

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

sc.settings.verbosity = 3

# Load gene resources

In [2]:
%%time 
# === Load and filter GTF ===
gtf_file = "/nfs/turbo/umms-indikar/shared/projects/reference_genome/prebuilt/refdata-gex-GRCh38-2024-A/genes/genes.gtf"
gtf = pr.read_gtf(gtf_file)

exons = gtf[gtf.Feature == "exon"].df
print(f"{exons.shape=}")
exons.head()


exons.shape=(1586950, 27)
CPU times: user 52 s, sys: 4.71 s, total: 56.7 s
Wall time: 56.9 s


,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_name,transcript_support_level,havana_transcript,exon_number,exon_id,exon_version,hgnc_id,havana_gene,protein_id,ccdsid
0,GL000009.2,ENSEMBL,exon,56139,58376,.,-,.,ENSG00000278704,1,...,ENST00000618686,NA,NaN,1,ENSE00003753029,1,NaN,NaN,ENSP00000484918.1,NaN
1,GL000194.1,ENSEMBL,exon,114985,115018,.,-,.,ENSG00000277400,1,...,ENST00000613230,1,NaN,1,ENSE00002299440,2,NaN,NaN,ENSP00000483280.1,NaN
2,GL000194.1,ENSEMBL,exon,112791,112850,.,-,.,ENSG00000277400,1,...,ENST00000613230,1,NaN,2,ENSE00003739295,1,NaN,NaN,ENSP00000483280.1,NaN
3,GL000194.1,ENSEMBL,exon,53589,55676,.,-,.,ENSG00000277400,1,...,ENST00000613230,1,NaN,3,ENSE00003723764,1,NaN,NaN,ENSP00000483280.1,NaN
4,GL000194.1,ENSEMBL,exon,114985,115055,.,-,.,ENSG00000274847,1,...,MAFIP-201,1,NaN,1,ENSE00003736481,1,HGNC:31102,NaN,ENSP00000478910.1,NaN


In [3]:
fpath = "../../resources/isoform_map.csv.gz"

idf = pd.read_csv(fpath)
print(f"{idf.shape=}")

idf['uniprot'] = np.where(idf['UniProtKB/TrEMBL ID'].isna(), idf['UniProtKB/Swiss-Prot ID'], idf['UniProtKB/TrEMBL ID'])

transcript_2_uniprot = dict(zip(idf['Transcript name'].values, idf['uniprot'].values))

idf.head()

idf.shape=(110149, 11)


,Gene stable ID,Gene stable ID version,Transcript stable ID,Transcript stable ID version,Protein stable ID,Protein stable ID version,Transcript length (including UTRs and CDS),Transcript name,Gene name,UniProtKB/Swiss-Prot ID,UniProtKB/TrEMBL ID,uniprot
0,ENSG00000198888,ENSG00000198888.2,ENST00000361390,ENST00000361390.2,ENSP00000354687,ENSP00000354687.2,956,MT-ND1-201,MT-ND1,P03886,U5Z754,U5Z754
1,ENSG00000198763,ENSG00000198763.3,ENST00000361453,ENST00000361453.3,ENSP00000355046,ENSP00000355046.4,1042,MT-ND2-201,MT-ND2,P03891,Q7GXY9,Q7GXY9
2,ENSG00000198804,ENSG00000198804.2,ENST00000361624,ENST00000361624.2,ENSP00000354499,ENSP00000354499.2,1542,MT-CO1-201,MT-CO1,P00395,U5YWV7,U5YWV7
3,ENSG00000198712,ENSG00000198712.1,ENST00000361739,ENST00000361739.1,ENSP00000354876,ENSP00000354876.1,684,MT-CO2-201,MT-CO2,P00403,U5Z487,U5Z487
4,ENSG00000228253,ENSG00000228253.1,ENST00000361851,ENST00000361851.1,ENSP00000355265,ENSP00000355265.1,207,MT-ATP8-201,MT-ATP8,P03928,U5YV54,U5YV54


In [4]:
fpath = "../../resources/allTFs_hg38.txt"
tf_list = [x.strip() for x in open(fpath)]
tf_list[:10]

['ZNF354C',
 'KLF12',
 'ZNF143',
 'ZIC2',
 'ZNF274',
 'SP2',
 'ZBTB7A',
 'BCL6B',
 'ZBTB49',
 'ZIC1']

# Load Data

In [5]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/isoforms.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 314 ms, sys: 548 ms, total: 862 ms
Wall time: 1.63 s


AnnData object with n_obs × n_vars = 21062 × 113335
    obs: 'dataset', 'dataset_bdata', 'source', 'cluster', 'group', 'label', 'filter_pass', 'bm_clusters', 'subgroup'
    var: 'gene_id', 'gene_name', 'gene_type', 'transcript_name', 'transcript_type', 'Chromosome', 'Start', 'End', 'transcript_id'
    uns: 'subgroup_colors'
    obsm: 'X_umap'

# Preprocessing

In [6]:
rsc.get.anndata_to_GPU(adata)
rsc.pp.filter_genes(adata, min_counts=30)

adata.layers['counts'] = adata.X.copy()

rsc.pp.normalize_total(adata, target_sum=1e4)
rsc.pp.log1p(adata)

adata.layers['log_norm'] = adata.X.copy()
adata

filtered out 43826 genes that are detected in less than 30 counts


AnnData object with n_obs × n_vars = 21062 × 69509
    obs: 'dataset', 'dataset_bdata', 'source', 'cluster', 'group', 'label', 'filter_pass', 'bm_clusters', 'subgroup'
    var: 'gene_id', 'gene_name', 'gene_type', 'transcript_name', 'transcript_type', 'Chromosome', 'Start', 'End', 'transcript_id', 'n_counts', 'n_cells'
    uns: 'subgroup_colors', 'log1p'
    obsm: 'X_umap'
    layers: 'counts', 'log_norm'

# Isoform Table

In [7]:
%%time 
adata.X = adata.layers['counts'].copy()
rsc.get.anndata_to_GPU(adata)
print(f"{type(adata.X)=}")

aggdata = rsc.get.aggregate(
    adata, 
    by='group',
    func='sum',
    axis=0,
)

rsc.get.anndata_to_CPU(aggdata, convert_all=True,)
df = aggdata.to_df(layer='sum')
df = df.T
df['total_transcript'] = df.sum(axis=1)

df = pd.merge(
    df, adata.var[['gene_name', 'gene_type', 'transcript_type']],
    how='left',
    left_index=True,
    right_index=True,
)
df = df.reset_index()
df = df.sort_values(by=['gene_name', 'transcript_name'])
df = df.reset_index(drop=True)
df['is_tf'] = df['gene_name'].isin(tf_list)
df['uniprot'] = df['transcript_name'].map(transcript_2_uniprot)
print(f"{df.shape=}")
df.head()

type(adata.X)=<class 'cupyx.scipy.sparse._csr.csr_matrix'>
df.shape=(69509, 11)
CPU times: user 217 ms, sys: 9.6 ms, total: 226 ms
Wall time: 230 ms


,transcript_name,bm_other,intial,reprogram,target,total_transcript,gene_name,gene_type,transcript_type,is_tf,uniprot
0,A1BG-202,26.0,10.0,36.0,6.0,78.0,A1BG,protein_coding,retained_intron,False,NaN
1,A1BG-203,31.0,34.0,104.0,8.0,177.0,A1BG,protein_coding,protein_coding_CDS_not_defined,False,NaN
2,A1BG-204,2909.0,8093.0,10093.0,628.0,21723.0,A1BG,protein_coding,retained_intron,False,NaN
3,A1BG-AS1-202,154.0,44.0,23.0,41.0,262.0,A1BG-AS1,lncRNA,lncRNA,False,NaN
4,A1BG-AS1-203,2.0,22.0,16.0,0.0,40.0,A1BG-AS1,lncRNA,lncRNA,False,NaN


In [8]:
df['transcript_name'].value_counts()

transcript_name
ZZZ3-211        1
A1BG-202        1
A1BG-203        1
A1BG-204        1
A1BG-AS1-202    1
               ..
A2M-201         1
A2M-208         1
A2M-AS1-201     1
A2M-AS1-203     1
A2M-AS1-204     1
Name: count, Length: 69509, dtype: int64

# add splicing events

In [9]:
fpath = "../../resources/AS_events.csv.gz"

asdf = pd.read_csv(fpath)
asdf = asdf[asdf['gene_name'].notna()]
asdf = asdf.drop(columns=['gene_id', 'transcript_id', 'gene_name'])
asdf.head()

df = pd.merge(
    df, asdf,
    how='left',
)
print(f"{df.shape=}")
print()

df.head()

df.shape=(69509, 19)



,transcript_name,bm_other,intial,reprogram,target,total_transcript,gene_name,gene_type,transcript_type,is_tf,uniprot,A3,A5,AF,AL,MX,RI,SE,num_events
0,A1BG-202,26.0,10.0,36.0,6.0,78.0,A1BG,protein_coding,retained_intron,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A1BG-203,31.0,34.0,104.0,8.0,177.0,A1BG,protein_coding,protein_coding_CDS_not_defined,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A1BG-204,2909.0,8093.0,10093.0,628.0,21723.0,A1BG,protein_coding,retained_intron,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A1BG-AS1-202,154.0,44.0,23.0,41.0,262.0,A1BG-AS1,lncRNA,lncRNA,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A1BG-AS1-203,2.0,22.0,16.0,0.0,40.0,A1BG-AS1,lncRNA,lncRNA,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Add metadata

In [10]:
%%time
pdf = df.copy()

# count the number of isoforms per gene
pdf['n_isoforms'] = pdf.groupby('gene_name')['transcript_name'].transform('nunique')
pdf['n_protiens'] = pdf.groupby('gene_name')['uniprot'].transform('nunique')

# add grouped expression counts
grouped = pdf.groupby('gene_name')[['bm_other', 'intial', 'reprogram', 'target']].sum()
sum_columns  = [f'{col}_sum' for col in grouped.columns]
grouped.columns = sum_columns
pdf = pdf.merge(grouped, on='gene_name', how='left')

pdf['total_gene'] = pdf[sum_columns].sum(axis=1)

# add percentages
for col in ['bm_other', 'intial', 'reprogram', 'target']:
    sum_col = f'{col}_sum'
    pct_col = f'{col}_pct'
    pdf[pct_col] = pdf[col] / pdf[sum_col]

print(pdf[['gene_name', 'transcript_name', 'intial_pct', 'reprogram_pct', 'target_pct']].head().to_string())

  gene_name transcript_name  intial_pct  reprogram_pct  target_pct
0      A1BG        A1BG-202    0.001229       0.003518    0.009346
1      A1BG        A1BG-203    0.004178       0.010163    0.012461
2      A1BG        A1BG-204    0.994593       0.986319    0.978193
3  A1BG-AS1    A1BG-AS1-202    0.156028       0.080702    0.546667
4  A1BG-AS1    A1BG-AS1-203    0.078014       0.056140    0.000000
CPU times: user 75.9 ms, sys: 1.05 ms, total: 77 ms
Wall time: 76.8 ms


In [12]:
def get_top(df, col):
    # Remove groups with any NaN
    mask = df.groupby('gene_name', observed=True)[col].transform(lambda x: x.isna().any())
    df_valid = df[~mask]

    # Get index of max value per group
    top_idx = df_valid.groupby('gene_name', observed=True)[col].idxmax()
    counts = df_valid.groupby('gene_name', observed=True)[col].apply(lambda x: (x == x.max()).sum())

    # Build mapping and drop ties
    top_map = df_valid.loc[top_idx, ['gene_name', 'transcript_name']].set_index('gene_name')['transcript_name']
    top_map[counts > 1] = np.nan

    return df['gene_name'].map(top_map)

for col in ['intial_pct', 'reprogram_pct', 'target_pct']:
    pdf[f'{col}_top'] = get_top(pdf, col)

pdf.head()

,transcript_name,bm_other,intial,reprogram,target,total_transcript,gene_name,gene_type,transcript_type,is_tf,...,reprogram_sum,target_sum,total_gene,bm_other_pct,intial_pct,reprogram_pct,target_pct,intial_pct_top,reprogram_pct_top,target_pct_top
0,A1BG-202,26.0,10.0,36.0,6.0,78.0,A1BG,protein_coding,retained_intron,False,...,10233.0,642.0,21978.0,0.008766,0.001229,0.003518,0.009346,A1BG-204,A1BG-204,A1BG-204
1,A1BG-203,31.0,34.0,104.0,8.0,177.0,A1BG,protein_coding,protein_coding_CDS_not_defined,False,...,10233.0,642.0,21978.0,0.010452,0.004178,0.010163,0.012461,A1BG-204,A1BG-204,A1BG-204
2,A1BG-204,2909.0,8093.0,10093.0,628.0,21723.0,A1BG,protein_coding,retained_intron,False,...,10233.0,642.0,21978.0,0.980782,0.994593,0.986319,0.978193,A1BG-204,A1BG-204,A1BG-204
3,A1BG-AS1-202,154.0,44.0,23.0,41.0,262.0,A1BG-AS1,lncRNA,lncRNA,False,...,285.0,75.0,958.0,0.487342,0.156028,0.080702,0.546667,A1BG-AS1-204,A1BG-AS1-209,A1BG-AS1-202
4,A1BG-AS1-203,2.0,22.0,16.0,0.0,40.0,A1BG-AS1,lncRNA,lncRNA,False,...,285.0,75.0,958.0,0.006329,0.078014,0.056140,0.000000,A1BG-AS1-204,A1BG-AS1-209,A1BG-AS1-202


In [11]:
pdf.head()

,transcript_name,bm_other,intial,reprogram,target,total_transcript,gene_name,gene_type,transcript_type,is_tf,...,n_protiens,bm_other_sum,intial_sum,reprogram_sum,target_sum,total_gene,bm_other_pct,intial_pct,reprogram_pct,target_pct
0,A1BG-202,26.0,10.0,36.0,6.0,78.0,A1BG,protein_coding,retained_intron,False,...,0,2966.0,8137.0,10233.0,642.0,21978.0,0.008766,0.001229,0.003518,0.009346
1,A1BG-203,31.0,34.0,104.0,8.0,177.0,A1BG,protein_coding,protein_coding_CDS_not_defined,False,...,0,2966.0,8137.0,10233.0,642.0,21978.0,0.010452,0.004178,0.010163,0.012461
2,A1BG-204,2909.0,8093.0,10093.0,628.0,21723.0,A1BG,protein_coding,retained_intron,False,...,0,2966.0,8137.0,10233.0,642.0,21978.0,0.980782,0.994593,0.986319,0.978193
3,A1BG-AS1-202,154.0,44.0,23.0,41.0,262.0,A1BG-AS1,lncRNA,lncRNA,False,...,0,316.0,282.0,285.0,75.0,958.0,0.487342,0.156028,0.080702,0.546667
4,A1BG-AS1-203,2.0,22.0,16.0,0.0,40.0,A1BG-AS1,lncRNA,lncRNA,False,...,0,316.0,282.0,285.0,75.0,958.0,0.006329,0.078014,0.056140,0.000000


# Prepare data

In [28]:
gene_list = [
    "MAP2K3", "ARRDC3", "NEK8", "WTIP", "CIT", "FRMD1", "SOX11",
    "MAPK14", "MOB3B", "FRMD6", "DLG5", "CORO7", "NF2", "SHANK2",
    "VGLL4", "SRC", "SCHIP1", "WWC1", "WWC2", "WWC3", "LIMD1",
    "IQCJ-SCHIP1", "TIAL1", "NUAK2", "MARK3"
]


columns = [
    'gene_name', 'transcript_name', 'transcript_type', 'uniprot',
     'intial_pct_top','reprogram_pct_top', 'target_pct_top',
]

qdf = pdf[columns].copy()
qdf = qdf[qdf['gene_name'].isin(gene_list)]
print(f"{qdf.shape=}")

# Vectorized approach
AB = qdf['intial_pct_top'] == qdf['reprogram_pct_top']
AC = qdf['intial_pct_top'] == qdf['target_pct_top']
BC = qdf['reprogram_pct_top'] == qdf['target_pct_top']

qdf['match_code'] = (
    AB.astype(int).astype(str) +
    AC.astype(int).astype(str) +
    BC.astype(int).astype(str)
)

# drop all matches and no matches
qdf = qdf[~qdf['match_code'].isin(['111', '000'])]
print(f"{qdf.shape=}")

qdf = qdf.sort_values('match_code')

print(qdf.head().to_string(index=False))

qdf.shape=(102, 7)
qdf.shape=(87, 8)
gene_name transcript_name         transcript_type uniprot intial_pct_top reprogram_pct_top target_pct_top match_code
    CORO7       CORO7-219          protein_coding  I3L258      CORO7-208         CORO7-201      CORO7-201        001
    TIAL1       TIAL1-213 nonsense_mediated_decay  E7ETC0      TIAL1-203         TIAL1-205      TIAL1-205        001
     DLG5        DLG5-206 nonsense_mediated_decay  R4GMQ2       DLG5-201          DLG5-202       DLG5-202        001
    TIAL1       TIAL1-203          protein_coding  Q01085      TIAL1-203         TIAL1-205      TIAL1-205        001
    TIAL1       TIAL1-205          protein_coding  Q01085      TIAL1-203         TIAL1-205      TIAL1-205        001


In [22]:
# 1. define your sectors (genes)
sectors = qdf['gene_name'].unique().tolist()

# 2. create the Circos object
circos = Circos(sectors=sectors)

# 3. for each pct‐top column, add one concentric track
radii = [
    (0.9, 1.0),   # outermost ring: initial_pct_top
    (0.8, 0.89),  # middle ring: reprogram_pct_top
    (0.7, 0.79),  # innermost ring: target_pct_top
]
cols = ['initial_pct_top', 'reprogram_pct_top', 'target_pct_top']

for (r0, r1), col in zip(radii, cols):
    track = circos.add_track((r0, r1))
    # set radial axis from 0 to 100%
    track.axis(range=(0, 100), ticks=[0, 50, 100])
    # draw one bar per sector
    for gene, sub in qdf.groupby('gene_name'):
        value = sub[col].astype(float).mean()  # or .iloc[0] if one row per gene
        track.bar(sector_id=gene, value=value)

# 4. render
circos.draw()
plt.tight_layout()
plt.show()


AttributeError: 'list' object has no attribute 'items'